# 4. BIKE SHARING - Preprocesamiento
**Tipo:** REGRESIÓN (predecir cantidad de alquileres)
**Variable objetivo:** count (total de bicicletas alquiladas)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# CARGAR DATOS
df = pd.read_csv('train.csv')
print(f"Dimensiones: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head()

In [ ]:
# ============================================================
# ⚠️ DATA LEAKAGE - ELIMINAR casual y registered
# ============================================================
# IMPORTANTE: En el test real NO tendrás estas columnas
# casual + registered = count (tu objetivo)
# Si las dejas, el modelo "hace trampa"

df = df.drop(columns=['casual', 'registered'])
print("✅ Columnas 'casual' y 'registered' eliminadas para evitar data leakage")

In [ ]:
# ============================================================
# INGENIERÍA DE CARACTERÍSTICAS - Procesar datetime
# ============================================================
# Extraer componentes útiles de la fecha/hora
df['datetime'] = pd.to_datetime(df['datetime'])

df['hora'] = df['datetime'].dt.hour
df['dia_semana'] = df['datetime'].dt.dayofweek  # 0=Lunes, 6=Domingo
df['mes'] = df['datetime'].dt.month
df['anio'] = df['datetime'].dt.year

# Eliminar datetime original (no sirve para el modelo)
df = df.drop(columns=['datetime'])
print(f"Nuevas columnas: {list(df.columns)}")

In [ ]:
# ============================================================
# ONE-HOT ENCODING para categóricas disfrazadas de números
# ============================================================
# season y weather son categóricas aunque sean números
# (4 no vale más que 1, son categorías distintas)

df = pd.get_dummies(df, columns=['season', 'weather'], drop_first=True)
print(f"Dimensiones después de encoding: {df.shape}")

In [ ]:
# ============================================================
# SEPARAR VARIABLE OBJETIVO
# ============================================================
y = df['count'].values
X_df = df.drop(columns=['count'])

print(f"Variable objetivo (y): {y.shape}")
print(f"Características: {X_df.shape}")
print(f"\nColumnas finales: {list(X_df.columns)}")

In [ ]:
# ============================================================
# DIVISIÓN DE DATOS (70/20/10)
# ============================================================
X = X_df.to_numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42)

print(f"Train: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val: {X_val.shape[0]}")
print(f"Test: {X_test.shape[0]}")

In [ ]:
# ============================================================
# NORMALIZACIÓN (solo variables continuas necesitan)
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("✅ Datos listos para REGRESIÓN")
print(f"Características: {X_train.shape[1]}")

## MÉTODO TRADICIONAL

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Regresión Lineal
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)

print("=== Regresión Lineal ===")
print(f"R² Score: {r2_score(y_test, y_pred_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.2f}")

# Random Forest (mejor para este problema)
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)

print("\n=== Random Forest ===")
print(f"R² Score: {r2_score(y_test, y_pred_rf):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.2f}")

## CON PYTORCH

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).reshape(-1, 1)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

In [ ]:
# Red Neuronal para Regresión
class RegresionNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.ReLU()  # ReLU al final porque no puede haber alquileres negativos
        )
    
    def forward(self, x):
        return self.red(x)

modelo = RegresionNN(X_train_t.shape[1])
criterio = nn.MSELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
print(modelo)